# Funnel Construction

## Objective

The cleaned e-commerce event dataset contains individual customer
interactions such as product views, cart additions, and purchases.

The objective of this notebook is to transform these event-level records
into a **user-level funnel dataset** that can be used for downstream
conversion and revenue analysis.

### Funnel

**Product View → Add to Cart → Purchase**

The funnel is constructed at the **unique-user level**.

### Output

The final user-level dataset contains one record per user with:

- Funnel stage indicators
- First view timestamp
- First qualifying cart timestamp
- First qualifying purchase timestamp
- Total purchase events
- Total purchase revenue

## 1. Funnel Definition & Business Rules

The funnel follows the chronological customer journey:

**View → Add to Cart → Purchase**

A user progresses through the funnel only when the required previous
stage has occurred.

### Funnel rules

1. A user must have a product view to qualify for the cart stage.
2. A cart event qualifies only when a view has already occurred.
3. A purchase qualifies for the funnel only when a qualifying cart event
   has already occurred.
4. Events must occur in chronological order.
5. Each user is represented by one record in the final funnel dataset.
6. The first qualifying timestamp is retained for each funnel stage.
7. All purchase events are retained separately for revenue analysis.

### Important distinction

The funnel `purchased` flag represents a **chronologically qualifying
purchase**.

`purchase_count` and `purchase_revenue`, however, represent all purchase
events associated with the user.

Therefore, users with purchase activity that does not satisfy the complete
View → Cart → Purchase sequence may still contribute to revenue analysis.

## 2. Funnel Construction Approach

The cleaned event dataset contains more than 42 million records, so it is
processed in chunks rather than loaded entirely into memory.

### Processing workflow

```text
Clean Event Data
       ↓
Read 500,000-row chunk
       ↓
Sort events chronologically
       ↓
Identify user-level view events
       ↓
Identify qualifying cart events
       ↓
Identify qualifying purchase events
       ↓
Update persistent user state
       ↓
Process next chunk
       ↓
Repeat until all events are processed
       ↓
Export one record per user

### Import Funnel Processing Function
The production funnel logic is implemented in `src/funnel_analysis.py`.
The notebook imports the function rather than duplicating the processing
logic inside the notebook.

In [1]:
import sys

sys.path.append("../src")

from funnel_analysis import build_user_funnel

## 3. Production Funnel Processing

The production pipeline processes the complete cleaned event dataset.

### Production configuration

- Input: `clean_events.csv`
- Chunk size: 500,000 rows
- Processing scope: Complete dataset
- Output: `user_funnel.csv`
- Granularity: One row per unique user

In [2]:
import time

start = time.time()

funnel = build_user_funnel(
    input_file="../data/processed/clean_events.csv",
    output_file="../data/processed/user_funnel.csv",
    db_file="../data/processed/user_funnel.db",
    chunk_size=500_000
    #max_chunks=1
)

elapsed = time.time() - start

print(f"\nProcessing time: {elapsed:.2f} seconds")
print(f"Processing time: {elapsed / 60:.2f} minutes")


Processing funnel chunk 1
Rows: 500,000
Users tracked so far: 89,167

Processing funnel chunk 2
Rows: 500,000
Users tracked so far: 163,082

Processing funnel chunk 3
Rows: 500,000
Users tracked so far: 228,077

Processing funnel chunk 4
Rows: 500,000
Users tracked so far: 295,993

Processing funnel chunk 5
Rows: 500,000
Users tracked so far: 350,866

Processing funnel chunk 6
Rows: 500,000
Users tracked so far: 410,771

Processing funnel chunk 7
Rows: 500,000
Users tracked so far: 461,888

Processing funnel chunk 8
Rows: 500,000
Users tracked so far: 513,155

Processing funnel chunk 9
Rows: 500,000
Users tracked so far: 568,440

Processing funnel chunk 10
Rows: 500,000
Users tracked so far: 612,121

Processing funnel chunk 11
Rows: 500,000
Users tracked so far: 661,574

Processing funnel chunk 12
Rows: 500,000
Users tracked so far: 710,346

Processing funnel chunk 13
Rows: 500,000
Users tracked so far: 753,414

Processing funnel chunk 14
Rows: 500,000
Users tracked so far: 799,882

P

### Production Run Result

The complete production dataset was processed successfully.

| Metric | Result |
|---|---:|
| Chunks processed | 85 |
| Event rows processed | 42,418,544 |
| Unique users | 3,022,290 |
| Chunk size | 500,000 |
| Processing time | 63.80 minutes |

The resulting dataset contains one record per unique user.

## 4. Production Output Validation

The generated `user_funnel.csv` is loaded and validated before being used
for downstream analysis.

In [6]:
print(funnel.shape)
print(funnel.head())
print(funnel.dtypes)

(3022290, 9)
     user_id  viewed  added_to_cart  purchased            first_view_time  \
0   33869381       1              0          0  2019-10-23T20:04:08+00:00   
1   64078358       1              0          0  2019-10-13T00:13:46+00:00   
2  183503497       1              0          0  2019-10-02T21:43:00+00:00   
3  184265397       1              0          0  2019-10-04T17:44:37+00:00   
4  195082191       1              0          0  2019-10-10T03:35:36+00:00   

  first_cart_time first_purchase_time  purchase_count  purchase_revenue  
0            None                None               0               0.0  
1            None                None               0               0.0  
2            None                None               0               0.0  
3            None                None               0               0.0  
4            None                None               0               0.0  
user_id                  int64
viewed                   int64
added_to_cart     

## 5. Final Funnel Results

The validated user-level dataset is aggregated to calculate the number of
unique users reaching each funnel stage.

In [7]:
print(
    funnel[
        [
            "viewed",
            "added_to_cart",
            "purchased"
        ]
    ].sum()
)

viewed           3022130
added_to_cart     336764
purchased         196488
dtype: int64


### Funnel Hierarchy Validation

A correctly constructed funnel must satisfy the following hierarchy:

**View ≥ Add to Cart ≥ Purchase**

The validation checks that:

- No user reaches cart without a qualifying view.
- No user reaches purchase without a qualifying cart.
- Cart users do not exceed view users.
- Purchase users do not exceed cart users.

In [8]:
# Users without a view
no_view = funnel[
    funnel["viewed"] == 0
]

print("Users without view:", len(no_view))

print(
    no_view[
        [
            "user_id",
            "viewed",
            "added_to_cart",
            "purchased",
            "purchase_count",
            "purchase_revenue"
        ]
    ]
)

Users without view: 160
           user_id  viewed  added_to_cart  purchased  purchase_count  \
859      402445064       0              0          0               0   
37658    512460500       0              0          0               1   
74942    512722770       0              0          0               0   
113021   512951247       0              0          0               1   
165632   513337572       0              0          0               1   
...            ...     ...            ...        ...             ...   
2999913  565987001       0              0          0               0   
3002881  566019381       0              0          0               3   
3003199  566023028       0              0          0               0   
3008441  566089642       0              0          0               0   
3020861  566245957       0              0          0               0   

         purchase_revenue  
859                  0.00  
37658              250.69  
74942                0.00  

In [9]:
# Check whether anyone reached cart without view
invalid_cart = funnel[
    (funnel["added_to_cart"] == 1) &
    (funnel["viewed"] == 0)
]

print(
    "Cart without view:",
    len(invalid_cart)
)

Cart without view: 0


In [10]:
# Check whether anyone reached purchase without cart
invalid_purchase = funnel[
    (funnel["purchased"] == 1) &
    (funnel["added_to_cart"] == 0)
]

print(
    "Purchase without qualifying cart:",
    len(invalid_purchase)
)

Purchase without qualifying cart: 0


In [11]:
# Check funnel hierarchy
print(
    "Cart > View:",
    (funnel["added_to_cart"] > funnel["viewed"]).sum()
)

print(
    "Purchase > Cart:",
    (funnel["purchased"] > funnel["added_to_cart"]).sum()
)

Cart > View: 0
Purchase > Cart: 0


In [12]:
funnel_counts = {
    "View": funnel["viewed"].sum(),
    "Add to Cart": funnel["added_to_cart"].sum(),
    "Purchase": funnel["purchased"].sum()
}

funnel_counts

{'View': np.int64(3022130),
 'Add to Cart': np.int64(336764),
 'Purchase': np.int64(196488)}

### Funnel Conversion Rates

Conversion rates are calculated using unique users rather than raw event
counts.

- **View → Cart** measures the percentage of viewed users who add a product
  to cart.
- **Cart → Purchase** measures the percentage of cart users who complete a
  qualifying purchase.
- **View → Purchase** measures the overall funnel conversion from viewing
  to qualifying purchase.

In [13]:
view_users = funnel["viewed"].sum()
cart_users = funnel["added_to_cart"].sum()
purchase_users = funnel["purchased"].sum()

view_to_cart = cart_users / view_users * 100
cart_to_purchase = purchase_users / cart_users * 100
view_to_purchase = purchase_users / view_users * 100

print(f"View → Cart:       {view_to_cart:.2f}%")
print(f"Cart → Purchase:   {cart_to_purchase:.2f}%")
print(f"View → Purchase:   {view_to_purchase:.2f}%")

View → Cart:       11.14%
Cart → Purchase:   58.35%
View → Purchase:   6.50%


### Funnel Construction Finding

The final user-level funnel contains:

- **3.02M** users who viewed products
- **336.8K** users who added products to cart
- **196.5K** users who completed a qualifying purchase

The funnel hierarchy validation returned zero violations, confirming that
users do not progress to later funnel stages without satisfying the required
previous stage.

## 6. Purchase-Only Users

A small number of users have purchase events but no recorded view or
qualifying cart in the dataset.

These users are not counted as funnel purchasers because they do not satisfy
the chronological View → Cart → Purchase definition.

They are retained in the user-level dataset because their purchase activity
and revenue remain relevant for revenue analysis.

In [ ]:
purchase_only_users = funnel[
    (funnel["viewed"] == 0) &
    (funnel["purchase_count"] > 0)
]

print("Purchase-only users:", len(purchase_only_users))

## 7. Output Dataset Description

The final `user_funnel.csv` contains one record per unique user.

| Column | Description |
|---|---|
| `user_id` | Unique user identifier |
| `viewed` | Whether the user reached the view stage |
| `added_to_cart` | Whether the user reached the qualifying cart stage |
| `purchased` | Whether the user reached the qualifying purchase stage |
| `first_view_time` | First recorded product-view timestamp |
| `first_cart_time` | First qualifying cart timestamp |
| `first_purchase_time` | First qualifying purchase timestamp |
| `purchase_count` | Total purchase events associated with the user |
| `purchase_revenue` | Total revenue from purchase events associated with the user |

This dataset serves as the primary user-level input for downstream funnel
and revenue analysis.